# 信噪 · GRPO 后训练（Colab）

在 SFT LoRA adapter 之上做 GRPO，使用混合 reward（LLM-as-judge 主体 + 规则项）。

## 准备
1. 把 SFT 训出来的 LoRA adapter 上传到 Google Drive（文件夹里有 `adapter_config.json` + `adapter_model.safetensors`）
2. Colab 左侧 🔑 Secrets 加 `DEEPSEEK_API_KEY`
3. Runtime → Change runtime type → GPU T4

**预估**: 65 prompts × 4 generations × 2 epochs ≈ 520 次 judge 调用，约 20-40 分钟，DeepSeek API < $1。

In [ ]:
# 1. 安装依赖（从零装，不用 unsloth，干净无冲突）
!pip install -q torch transformers peft accelerate datasets trl bitsandbytes
!pip install -q openai python-dotenv

In [ ]:
# 2. 挂载 Drive + 克隆代码 + 注入 API Key
import os
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive")

# 注入 API Key（Colab 左侧 🔑 Secrets 里添加 DEEPSEEK_API_KEY）
os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")

# 克隆最新代码
WORK_DIR = Path("/content/repo")
if WORK_DIR.exists():
    import shutil; shutil.rmtree(WORK_DIR)
!git clone https://github.com/alchosyn/npc-dialogue-ai-agent.git {WORK_DIR}
os.chdir(WORK_DIR)

# 生成训练数据
!python scripts/build_grpo_dataset.py
TRAIN_JSONL = WORK_DIR / "data" / "grpo_train.jsonl"

# ===== 改这里：指向你 Drive 里的 SFT adapter 路径 =====
SFT_ADAPTER = Path("/content/drive/MyDrive/xinzao-lora-adapter")
# =====================================================

assert (SFT_ADAPTER / "adapter_config.json").exists(), \
    f"找不到 adapter_config.json，检查路径: {SFT_ADAPTER}"

OUTPUT_DIR = Path("/content/qwen-1.5b-xinzao-grpo")
print(f"训练数据: {TRAIN_JSONL}")
print(f"SFT adapter: {SFT_ADAPTER}")
print(f"输出: {OUTPUT_DIR}")

import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} (bf16: {torch.cuda.is_bf16_supported()})")

In [ ]:
# 3. 跑 GRPO 训练
import subprocess, sys
cmd = [
    sys.executable, "scripts/train_grpo.py",
    "--train-jsonl", str(TRAIN_JSONL),
    "--sft-adapter", str(SFT_ADAPTER),
    "--output-dir", str(OUTPUT_DIR),
    "--no-unsloth",
    "--num-generations", "4",
    "--batch-size", "4",
    "--grad-accum", "4",
    "--max-prompt-length", "384",
    "--max-completion-length", "384",
    "--num-train-epochs", "2",
    "--learning-rate", "1e-6",
    "--judge-workers", "8",
    "--precision", "auto",
]
print("+ " + " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# 4. 保存到 Drive + 看输出
import shutil
DRIVE_OUTPUT = Path("/content/drive/MyDrive/qwen-1.5b-xinzao-grpo")
if DRIVE_OUTPUT.exists():
    shutil.rmtree(DRIVE_OUTPUT)
shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT)
!ls -lh {OUTPUT_DIR}
print(f"\n已保存到 Drive: {DRIVE_OUTPUT}")
print("下一步：开 eval notebook 跑 5 路对比")